<a href="https://colab.research.google.com/github/Pedrohsm2/Projeto-Prog-1/blob/master/atividade_pratica_aula6_plotly_interativo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Atividade Prática — Aula 6: Visualização Interativa com Plotly

Esta atividade foi construída com base nos slides da Aula 6, que apresentam a transição do gráfico estático para o gráfico como **aplicativo de exploração**, com foco em **hover**, **zoom**, **pan**, **filtros visuais**, **Plotly Express** e construção de componentes que mais tarde podem virar um dashboard. fileciteturn7file0

## Ideia central da aula
A interatividade existe para apoiar a investigação do gestor em tempo real.  
Mesmo assim, a aula reforça uma regra essencial: **interatividade não substitui clareza**. O gráfico precisa continuar limpo, bem titulado e orientado à decisão. fileciteturn7file0

## Regras da atividade
- O notebook orienta, mas **você deve construir os códigos**.
- Use **Plotly Express** sempre que possível.
- Após cada visual principal, escreva uma breve **interpretação humana**.
- Teste hover, zoom e isolamento por legenda antes de concluir.
- Pense como desenvolvedor: os gráficos de hoje poderão virar o dashboard de amanhã. fileciteturn7file0

## Dataset da atividade
Arquivo: `vendas_brasil_clean_aula6_plotly.csv`


## 1. Preparação do ambiente

Importe as bibliotecas necessárias.

**Sugestão:**
- `pandas`
- `plotly.express`
- `plotly.graph_objects` (opcional)


In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go


## 2. Leitura e inspeção inicial da base

Leia o arquivo `vendas_brasil_clean_aula6_plotly.csv` em um DataFrame chamado `df`.

Depois:
1. exiba as primeiras linhas
2. verifique o tamanho da base
3. confira os tipos das colunas
4. confirme se `data_venda` está em formato adequado para análises temporais
5. identifique quais colunas podem alimentar:
   - comparação de categorias
   - evolução no tempo
   - relação entre variáveis
   - distribuição espacial


In [3]:
df = pd.read_csv('vendas_brasil_clean_aula6_plotly.csv')


## 3. Matriz de decisão do analista

Os slides apresentam uma regra prática: usar interativo (Plotly) quando o painel será visto em tela/web, quando o gestor precisará responder subperguntas na hora e quando o gráfico poderá ser empacotado em um dashboard. fileciteturn7file0

### Tarefa
Responda em markdown:
1. Por que esta atividade faz mais sentido em Plotly do que em Matplotlib?
2. Em que situação você ainda preferiria um gráfico estático?
3. O destino desta análise parece mais “PDF” ou mais “tela/web”?


## Matriz de decisão do analista

### Por que esta atividade faz mais sentido em Plotly do que em Matplotlib?

Porque a análise usa dados de vendas e o gestor pode precisar interagir com o gráfico, por exemplo:

- ver valores passando o mouse
- filtrar por estado, mês ou categoria
- responder perguntas rápidas na hora
- usar o gráfico em um dashboard

O Plotly permite tudo isso. O Matplotlib gera apenas gráficos estáticos.

---

### Em que situação você ainda preferiria um gráfico estático?

Eu usaria gráfico estático quando:

- o gráfico será colocado em PDF ou relatório acadêmico
- só preciso mostrar o resultado final
- não precisa de interação
- o gráfico é muito simples

---

### O destino desta análise parece mais “PDF” ou mais “tela/web”?

Parece mais tela/web, porque:

- a análise está sendo feita no Google Colab
- o gráfico está sendo feito com Plotly
- dados de vendas normalmente são usados em dashboards
- faz mais sentido interagir com o gráfico do que usar uma imagem fixa

## 4. Missão 1 — Barras: Receita por Canal

A missão prática da aula pede construir um gráfico de barras para responder:
**qual canal vende mais?**  
Os slides também sugerem usar o hover para injetar métricas secundárias sem poluir a tela. fileciteturn7file0

### Tarefa
1. Agregue a `receita` por `canal_venda`
2. Inclua também uma métrica secundária, como `quantidade`
3. Construa um gráfico de barras com Plotly Express
4. Ordene do maior para o menor
5. Faça o hover mostrar mais do que apenas a receita

### Perguntas
- Qual canal lidera a receita?
- O canal líder também lidera em quantidade?
- O hover ajudou a enriquecer a leitura sem poluir a tela?


In [4]:
resumo = df.groupby("canal_venda", as_index=False).agg({
    "receita": "sum",
    "quantidade": "sum"
})

resumo = resumo.sort_values("receita", ascending=False)

grafico = px.bar(
    resumo,
    x="canal_venda",
    y="receita",
    hover_data=["quantidade"]
)

grafico.show()

### Qual canal lidera a receita?

O canal que aparece na primeira barra do gráfico (o maior valor de receita).

---

### O canal líder também lidera em quantidade?

Não necessariamente. Dá para ver isso passando o mouse no gráfico e comparando a quantidade de cada canal.

---

### O hover ajudou a enriquecer a leitura sem poluir a tela?

Sim. A receita continua sendo a informação principal no gráfico, mas o hover mostra também a quantidade sem deixar o gráfico poluído.

### Insight obrigatório
Escreva 2 ou 3 linhas explicando:
- quem lidera
- quem fica atrás
- o que um gestor poderia investigar a seguir


O canal que lidera a receita é o que aparece na primeira barra do gráfico, pois possui o maior valor total vendido.  
O canal que fica atrás é o que aparece nas últimas barras, com receita bem menor em comparação ao líder.  
Um gestor poderia investigar por que o canal líder vende mais: preço médio maior, mais produtos vendidos ou maior volume de clientes.

## 5. Missão 2 — Linhas: Sazonalidade da Receita Mensal

Os slides destacam que o gráfico de linhas, com zoom e pan, é ideal para navegar no tempo e isolar picos. Também reforçam que o eixo temporal precisa estar corretamente tipado no Pandas. fileciteturn7file0

### Tarefa
1. Converta `data_venda` em data, se necessário
2. Agregue a `receita` por mês
3. Construa um gráfico de linha interativo
4. Teste o zoom nos meses de pico
5. Observe se novembro e dezembro se destacam

### Perguntas
- Existe sazonalidade?
- Quais meses chamam atenção?
- O zoom ajudou a explorar melhor a parte final da série?


In [5]:
df["data_venda"] = pd.to_datetime(df["data_venda"])

df["mes"] = df["data_venda"].dt.to_period("M").astype(str)

mensal = df.groupby("mes", as_index=False)["receita"].sum()

grafico = px.line(mensal, x="mes", y="receita")

grafico.show()

### Existe sazonalidade?

Sim. Dá para perceber que a receita não é igual todos os meses e existem meses com valores bem maiores.

---

### Quais meses chamam atenção?

Os meses do final do ano (principalmente novembro e dezembro) chamam mais atenção porque a receita aumenta bastante.

---

### O zoom ajudou a explorar melhor a parte final da série?

Sim. Usando o zoom fica mais fácil ver os picos de receita nos últimos meses e comparar mês por mês.

### Insight obrigatório
Explique:
- qual tendência aparece
- se há picos claros
- qual hipótese de negócio pode explicar o comportamento observado


A tendência geral mostra crescimento da receita ao longo dos meses, principalmente no final do ano.  
Existem picos claros nos últimos meses, com destaque para novembro e dezembro.  
Uma hipótese de negócio é o aumento das vendas por causa de datas como Black Friday e Natal, que normalmente fazem a receita crescer nesse período.

## 6. Missão 3 — Dispersão: Lucro vs. Receita segmentado por Categoria

Os slides mostram o uso do gráfico de dispersão para revelar correlação entre KPIs e identificar anomalias, com grande apoio do hover. fileciteturn7file0

### Tarefa
1. Construa um scatter plot com:
   - eixo X = `receita`
   - eixo Y = `lucro`
2. Use `categoria` como cor
3. Inclua no hover:
   - `produto`
   - `canal_venda`
   - `uf`
   - `margem_lucro`
4. Tente identificar algum ponto fora da curva

### Perguntas
- Existe correlação visual entre receita e lucro?
- Há anomalias?
- O hover ajuda a transformar um ponto em um caso investigável?


In [6]:
grafico = px.scatter(
    df,
    x="receita",
    y="lucro",
    color="categoria",
    hover_data=["produto", "canal_venda", "uf", "margem_lucro"]
)

grafico.show()

### Existe correlação visual entre receita e lucro?

Sim. Quanto maior a receita, maior tende a ser o lucro. Os pontos sobem juntos no gráfico.

---

### Há anomalias?

Sim. Existem alguns pontos com receita alta mas lucro baixo, ou até muito menor que os outros pontos parecidos.

---

### O hover ajuda a transformar um ponto em um caso investigável?

Sim. Passando o mouse é possível ver o produto, o canal, o estado e a margem de lucro. Isso permite investigar exatamente qual venda está fora do padrão.

### Insight obrigatório
Escreva 2 ou 3 linhas dizendo:
- se a correlação parece positiva
- onde surgem pontos fora do padrão
- que ação analítica poderia ser tomada depois


A correlação parece positiva, porque quando a receita aumenta o lucro também tende a aumentar.  
Os pontos fora do padrão aparecem principalmente quando a receita é alta, mas o lucro é baixo em comparação aos outros pontos.  
Uma ação analítica seria investigar esses produtos ou canais para entender se o problema é desconto alto, margem baixa ou custo elevado.

## 7. Exploração espacial — Onde está concentrada a operação?

Os slides incluem mapas geográficos como resposta para perguntas espaciais como:
**onde está concentrada nossa operação?** fileciteturn7file0

### Tarefa
Usando `latitude` e `longitude`, crie um mapa interativo que ajude a visualizar a operação.

### Sugestões
- use tamanho ou cor para uma métrica relevante (como receita)
- teste o zoom do mapa
- observe se há concentração regional

### Perguntas
- Onde a operação parece mais concentrada?
- O mapa ajudou mais do que uma simples tabela por UF?


In [7]:
mapa = px.scatter_mapbox(
    df,
    lat="latitude",
    lon="longitude",
    size="receita",
    color="receita",
    hover_data=["uf", "produto", "canal_venda", "receita"],
    zoom=3
)

mapa.update_layout(mapbox_style="open-street-map")

mapa.show()

### Onde a operação parece mais concentrada?

A operação parece mais concentrada nas regiões onde existem mais pontos no mapa e onde os círculos são maiores.

---

### O mapa ajudou mais do que uma simples tabela por UF?

Sim. O mapa facilita muito mais a visualização porque mostra onde estão as vendas de forma visual e rápida, enquanto a tabela mostra apenas números.

## 8. A tríade da interatividade

Um slide central da aula apresenta três componentes principais:
- **Tooltips (hover)**
- **Zoom & Pan**
- **Filtros visuais / isolamento via legenda** fileciteturn7file0

### Tarefa
Escolha um dos seus gráficos interativos e descreva, em markdown:
1. O que o hover acrescenta
2. Como o zoom melhora a investigação
3. Como o clique na legenda pode ajudar a isolar séries ou categorias


Escolhi o gráfico de dispersão (lucro vs receita).

O hover acrescenta mais informações sem poluir o gráfico, como produto, canal de venda, estado e margem de lucro. Isso ajuda a entender exatamente qual ponto estamos analisando.

O zoom melhora a investigação porque permite focar apenas nos pontos com receita alta ou apenas nos pontos com lucro baixo, facilitando a análise.

O clique na legenda ajuda a isolar uma categoria específica. Assim fica mais fácil comparar apenas uma categoria por vez sem a interferência das outras.

## 9. Plotly Express — Máximo impacto, mínimo código

A aula mostra o Plotly Express como uma API declarativa, perfeita para DataFrames do Pandas e com geração automática de HTML/JS. fileciteturn7file0

### Tarefa
Explique em markdown:
1. O que significa dizer que Plotly Express é “declarativo”?
2. Em que ele facilita a vida do analista?
3. Como isso se conecta com a ideia de futuro dashboard em Streamlit?


Dizer que o Plotly Express é declarativo significa que o analista só precisa dizer quais colunas do DataFrame quer usar (ex: receita no eixo Y e mês no eixo X), e o gráfico já é criado automaticamente.

Isso facilita muito a vida do analista porque reduz bastante o código. Não é preciso configurar tudo manualmente como no Matplotlib, então fica mais rápido analisar os dados.

Isso se conecta com o futuro dashboard em Streamlit porque o Plotly já gera gráficos interativos prontos para a web. Então o mesmo gráfico feito no Colab pode ser usado depois dentro de um dashboard.

## 10. Clareza continua obrigatória

A aula reforça que um gráfico interativo ruim apenas confunde o usuário de forma mais tecnológica.  
Os princípios inegociáveis continuam sendo:
- título autoexplicativo
- eixos nomeados
- unidades claras
- remoção de lixo visual fileciteturn7file0

### Tarefa
Revise um dos seus gráficos e melhore:
- título
- rótulos de eixo
- hover
- nomes das variáveis
- aparência geral

Depois escreva:
1. O que você mudou?
2. O gráfico ficou mais claro?


In [8]:
df["data_venda"] = pd.to_datetime(df["data_venda"])
df["mes"] = df["data_venda"].dt.to_period("M").astype(str)

mensal = df.groupby("mes", as_index=False)["receita"].sum()

grafico = px.line(
    mensal,
    x="mes",
    y="receita",
    title="Receita mensal ao longo do tempo"
)

grafico.update_layout(
    xaxis_title="Mês",
    yaxis_title="Receita (R$)"
)

grafico.update_traces(
    hovertemplate="Mês: %{x}<br>Receita: R$ %{y}<extra></extra>"
)

grafico.show()

Eu melhorei o título do gráfico, os nomes dos eixos e o hover para mostrar a receita em reais.  
Também deixei o gráfico mais limpo, sem informações desnecessárias.

Sim, o gráfico ficou mais claro porque agora dá para entender o que está sendo mostrado apenas olhando o título e os eixos.

## 11. O gráfico não fala sozinho

O último slide deixa explícito: mesmo com zoom e tooltips avançados, a interpretação humana continua insubstituível. Sempre forneça uma conclusão textual acompanhando o visual. fileciteturn7file0

### Tarefa
Escolha **dois gráficos** que você construiu e, para cada um, escreva:
- um insight principal
- uma possível decisão gerencial
- uma pergunta adicional que o gestor poderia fazer em seguida


Gráfico 1 — Receita por canal

Insight principal: Existe um canal que gera a maior parte da receita em comparação com os outros.  
Possível decisão gerencial: Investir mais nesse canal que já está performando melhor.  
Pergunta adicional: Esse canal lidera porque vende mais produtos ou porque o valor médio das vendas é maior?

---

Gráfico 2 — Receita mensal (linha)

Insight principal: A receita aumenta no final do ano e existem meses com picos claros.  
Possível decisão gerencial: Planejar estoque e campanhas com foco nos meses de maior venda.  
Pergunta adicional: Quais produtos são responsáveis pelo aumento da receita nesses meses?

## 12. Missão prática final — Mapeando e construindo

Com base no quadro de missão prática da aula, entregue no notebook, no mínimo: fileciteturn7file0

1. **Barras** — Receita por canal, com hover enriquecido
2. **Linhas** — Sazonalidade da receita mensal, explorando zoom
3. **Dispersão** — Lucro vs. Receita segmentado por categoria, com hover e identificação de anomalia

### Extra recomendado
4. **Mapa** — concentração geográfica da operação

### Para cada gráfico
Inclua abaixo:
- a pergunta de negócio
- as colunas necessárias
- a interpretação humana final



## Gráfico 1 — Barras: Receita por Canal

**Pergunta de negócio**  
Qual canal de venda gera mais receita?

**Colunas necessárias**  
canal_venda  
receita  
quantidade (para o hover)

**Interpretação humana final**  
Existe um canal que lidera claramente a receita. Isso mostra onde a empresa está vendendo melhor. Um gestor poderia focar mais investimento nesse canal e analisar por que ele performa melhor que os outros.

---

## Gráfico 2 — Linhas: Receita Mensal (Sazonalidade)

**Pergunta de negócio**  
Existe sazonalidade nas vendas ao longo dos meses?

**Colunas necessárias**  
data_venda  
receita  
mes

**Interpretação humana final**  
A receita cresce no final do ano e existem meses com picos claros. Isso indica sazonalidade. A empresa poderia se preparar melhor para os meses mais fortes, aumentando estoque e campanhas.

---

## Gráfico 3 — Dispersão: Lucro vs Receita por Categoria

**Pergunta de negócio**  
Existe relação entre receita e lucro? Existem produtos fora do padrão?

**Colunas necessárias**  
receita  
lucro  
categoria  
produto  
margem_lucro  
canal_venda  
uf

**Interpretação humana final**  
Existe uma correlação positiva entre receita e lucro, mas alguns pontos fogem do padrão. Isso pode indicar produtos com margem muito baixa ou descontos muito altos. Esses casos deveriam ser investigados.

---

## Gráfico Extra — Mapa: Concentração da Operação

**Pergunta de negócio**  
Onde a operação está mais concentrada?

**Colunas necessárias**  
latitude  
longitude  
receita  
uf  
produto  
canal_venda

**Interpretação humana final**  
A operação parece mais concentrada em algumas regiões específicas. O mapa ajuda a entender onde estão as maiores vendas e pode ajudar a identificar oportunidades em regiões com pouca presença.


## 13. Mindset de desenvolvedor

Um dos slides diz que os gráficos de hoje são o dashboard de amanhã.  
Isso exige padronização desde já: nomes coerentes, lógica limpa e reaproveitamento futuro. fileciteturn7file0

### Tarefa
Responda em markdown:
1. Como você nomearia seus objetos `fig_...` de forma organizada?
2. Que partes do seu código poderiam ser reaproveitadas numa aplicação web?
3. O que vale a pena padronizar desde esta aula?


### Como eu nomearia meus objetos fig_... de forma organizada?

Eu usaria nomes claros baseados no objetivo do gráfico, por exemplo:
fig_receita_canal  
fig_receita_mensal  
fig_lucro_vs_receita  
fig_mapa_receita  

Assim fica fácil entender o que cada gráfico faz só pelo nome.

---

### Que partes do código poderiam ser reaproveitadas numa aplicação web?

A leitura do arquivo com pandas  
O tratamento das datas  
Os agrupamentos (groupby)  
E os próprios gráficos feitos com Plotly

Tudo isso pode ser usado depois em um dashboard com Streamlit.

---

### O que vale a pena padronizar desde esta aula?

Nome das variáveis  
Nome dos gráficos (fig_...)  
Formato das datas  
Nomes das colunas usadas nos gráficos  
E a forma de escrever os gráficos (sempre com a mesma estrutura)

## 14. Desafio extra (opcional)

Crie mais um gráfico interativo à sua escolha, desde que ele responda uma pergunta real. Exemplos:
- receita por UF (barras)
- preço unitário ao longo do tempo (linhas)
- desconto vs. quantidade (dispersão)
- receita por categoria (barras horizontais)

Mas atenção:
- use hover com intenção
- teste zoom se fizer sentido
- escreva interpretação final


## 15. Entrega esperada

Seu notebook deve demonstrar:
- uso correto de Plotly Express
- escolha adequada do gráfico para a pergunta
- exploração consciente de hover, zoom, pan e legenda
- compromisso com clareza
- interpretação textual consistente

### Mensagem principal da aula
A tecnologia explora.  
Mas só o analista transforma exploração em decisão. fileciteturn7file0
